In [ ]:
import sys
!"{sys.executable}" -m pip install matplotlib numpy torch datasets pyarrow huggingface_hub
import random
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active compute device: {device}")

# Load Dataset
ds = load_dataset("papluca/language-identification")


In [ ]:
#pipeline
train_labels_raw = [item["labels"] for item in ds["train"]]
label_counts = Counter(train_labels_raw)
# Label Mapping Construction
unique_labels = sorted(list(label_counts.keys()))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_classes = len(label2id)


def tokenize(text):
    return text.lower().split()

token_counter = Counter()
for item in ds["train"]:
    token_counter.update(tokenize(item["text"]))

VOCAB_CAP = 40000
vocab = {"<PAD>": 0, "<UNK>": 1}

most_common = token_counter.most_common(VOCAB_CAP - 2)
for token, _ in most_common:
    vocab[token] = len(vocab)

vocab_size = len(vocab)
pad_idx = vocab["<PAD>"]
unk_idx = vocab["<UNK>"]


max_len = 64

def encode_sequence(text):
    tokens = tokenize(text) if isinstance(text, str) else text
    ids = [vocab.get(token, unk_idx) for token in tokens[:max_len]]
    if len(ids) < max_len:
        ids = ids + [pad_idx] * (max_len - len(ids))
    return ids

def collate_batch(batch):
    input_ids = []
    target_labels = []
    for item in batch:
        input_ids.append(encode_sequence(item["text"]))
        target_labels.append(label2id[item["labels"]])
    return (
        torch.tensor(input_ids, dtype=torch.long),
        torch.tensor(target_labels, dtype=torch.long)
    )


BATCH_SIZE = 64
train_loader = DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader   = DataLoader(ds["validation"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader  = DataLoader(ds["test"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

In [ ]:
#Architecture
embed_dim = 128
hidden_dim = 256

embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx).to(device)
lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True).to(device)
dropout = nn.Dropout(0.3).to(device)
fc = nn.Linear(hidden_dim * 2, num_classes).to(device)

def forward_pass(x):
    embedded = dropout(embedding(x))
    _, (hidden, _) = lstm(embedded)
    hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
    logits = fc(dropout(hidden_cat))
    return logits

all_params = list(embedding.parameters()) + list(lstm.parameters()) + list(fc.parameters())
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(all_params, lr=1e-3)

In [ ]:
#training and evaluation
EPOCHS = 5

for epoch in range(EPOCHS):
    embedding.train(); lstm.train(); dropout.train(); fc.train()
    running_train_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = forward_pass(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    
    embedding.eval(); lstm.eval(); dropout.eval(); fc.eval()
    running_val_loss = 0.0
    correct_val = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = forward_pass(inputs)
            loss = criterion(outputs, targets)
            running_val_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct_val += (preds == targets).sum().item()
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_val_acc = (correct_val / len(val_loader.dataset)) * 100
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")


embedding.eval(); lstm.eval(); dropout.eval(); fc.eval()
test_loss = 0.0
test_correct = 0

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = forward_pass(inputs)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        test_correct += (preds == targets).sum().item()

final_test_loss = test_loss / len(test_loader.dataset)
final_test_acc = (test_correct / len(test_loader.dataset)) * 100

print("\n" + "="*40)
print(f"FINAL TEST RESULTS ({len(test_loader.dataset)} samples)")
print("="*40)
print(f"Test Loss    : {final_test_loss:.4f}")
print(f"Test Accuracy: {final_test_acc:.2f}%")

In [ ]:
#Custom Inference Pipeline
def predict_language(raw_text):
    embedding.eval(); lstm.eval(); dropout.eval(); fc.eval()
    
    encoded = encode_sequence(raw_text)
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = forward_pass(input_tensor)
        probabilities = torch.softmax(logits, dim=1).squeeze(0)
        
    predicted_id = torch.argmax(probabilities).item()
    predicted_label = id2label[predicted_id]
    confidence = probabilities[predicted_id].item()
    
    top3_probs, top3_indices = torch.topk(probabilities, 3)
    top3_results = [(id2label[idx.item()], prob.item()) for idx, prob in zip(top3_indices, top3_probs)]
    
    return {
        "text": raw_text,
        "predicted_language": predicted_label,
        "confidence": confidence,
        "top_3_predictions": top3_results
    }

# Example Inference Test
sample_msg = "ذهبو لكي يشترون بعض الاغراض للمنزل والدتهم كانت تطبخ ."
print("\nInference Output:", predict_language(sample_msg))